# Recommender Evaluation
This notebook compares CalCourse's TF-IDF, semantic, and hybrid ranking approaches using manually labeled student profiles and ranking metrics.

In [77]:
import pandas as pd

courses = pd.read_csv(
    "../data/processed/recommendable_courses_fall_2026.csv"
)

courses["course"] = (
    courses["subject"] + " " + courses["course_number"].astype(str)
)

In [78]:
evaluation_profiles = [
    {
        "name": "Machine Learning / Data Science",
        "interests": "machine learning, data science, predictive modeling, statistics",
        "preferred_subjects": ["DATA", "STAT", "COMPSCI", "INDENG"],
        "relevant_courses": [
            "DATA C102",
            "DATA 145",
            "STAT 154",
            "STAT 159",
            "INDENG 142A",
            "COMPSCI 189",
        ],
    },

    {
        "name": "Product Analytics / PM",
        "interests": "product management, experimentation, customer behavior, marketing analytics",
        "preferred_subjects": ["UGBA", "DATA", "INDENG", "ENGIN"],
        "relevant_courses": [
            "UGBA 104",
            "UGBA 160",
            "UGBA 161",
            "UGBA 162",
            "INDENG 142A",
            "ENGIN 183D",
        ],
    },

    {
        "name": "Economics / Public Policy",
        "interests": "economics, public policy, causal inference, inequality, labor markets",
        "preferred_subjects": ["ECON", "PUBPOL", "STAT", "DATA"],
        "relevant_courses": [
            "ECON 130",
            "ECON 140",
            "ECON 141",
            "ECON 151",
            "PUBPOL 141",
            "ECON 121",
        ],
    },

    {
        "name": "Healthcare / Computational Biology",
        "interests": "healthcare, computational biology, biomedical data, machine learning",
        "preferred_subjects": ["CMPBIO", "BIOENG", "DATA", "STAT"],
        "relevant_courses": [
            "CMPBIO 175",
            "BIOENG 140L",
            "STAT 159",
            "DATA 145",
        ],
    },

    {
        "name": "Engineering Systems / Optimization",
        "interests": "optimization, simulation, engineering systems, applied mathematics",
        "preferred_subjects": ["INDENG", "ENGIN", "MATH", "AEROENG", "CIVENG"],
        "relevant_courses": [
            "INDENG 174",
            "MATH 170",
            "AEROENG C144",
            "CHMENG 130",
            "ELENG 66",
        ],
    },

    {
        "name": "Software / Systems",
        "interests": "databases, distributed systems, backend engineering, operating systems",
        "preferred_subjects": ["COMPSCI", "EECS"],
        "relevant_courses": [
            "COMPSCI 162",
            "COMPSCI 169A",
            "COMPSCI 186",
            "COMPSCI 161",
        ],
    },

    {
        "name": "Statistics",
        "interests": "statistical modeling, probability, inference, time series",
        "preferred_subjects": ["STAT", "DATA", "MATH"],
        "relevant_courses": [
            "STAT 133",
            "STAT 134",
            "STAT 135",
            "STAT 151A",
            "STAT 153",
            "STAT 154",
            "STAT 159",
        ],
    },

    {
        "name": "Finance / Quantitative Analysis",
        "interests": "finance, quantitative analysis, risk, forecasting, markets",
        "preferred_subjects": ["ECON", "STAT", "INDENG", "UGBA"],
        "relevant_courses": [
            "ECON 136",
            "STAT 153",
            "INDENG 172",
            "UGBA 103",
        ],
    },

    {
        "name": "Behavioral Science",
        "interests": "decision making, psychology, behavioral economics, human behavior",
        "preferred_subjects": ["PSYCH", "COGSCI", "ECON", "PUBPOL"],
        "relevant_courses": [
            "COGSCI 151",
            "PUBPOL 141",
            "ECON 119",
        ],
    },

    {
        "name": "Design / HCI",
        "interests": "human computer interaction, product design, user research, interface design",
        "preferred_subjects": ["DESINV", "INFO", "COGSCI", "ENGIN"],
        "relevant_courses": [
            "ENGIN 183D",
        ],
    },
]

Relevance labels were expanded using profile-specific criteria rather than model outputs, reducing the risk of penalizing valid recommendations simply because the original label set was too sparse.

In [79]:
course_labels = set(courses["course"])

for profile in evaluation_profiles:
    missing = [
        course
        for course in profile["relevant_courses"]
        if course not in course_labels
    ]

    if missing:
        print(profile["name"], "missing:", missing)

## 2. Ranking Functions
Each student proile is ranked using three approaches: 

- TF-IDF keyword similarity 
- Semantic embedding similarity
- Hybrid ranking that combines semantic relevance with subject preferences

In [80]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

In [81]:
courses["text"] = (
    courses["title"].fillna("") + ". " +
    courses["description"].fillna("")
)

tfidf_vectorizer = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    courses["text"]
)

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

course_embeddings = embedding_model.encode(
    courses["text"].tolist(),
    show_progress_bar=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/65 [00:00<?, ?it/s]

In [82]:
def rank_tfidf(profile, top_k=10):
    profile_vector = tfidf_vectorizer.transform(
        [profile["interests"]]
    )

    scores = cosine_similarity(
        profile_vector,
        tfidf_matrix
    ).flatten()

    ranked = courses.copy()
    ranked["score"] = scores

    return ranked.sort_values(
        "score",
        ascending=False
    ).head(top_k)

In [83]:
def rank_semantic(profile, top_k=10):
    profile_embedding = embedding_model.encode(
        [profile["interests"]]
    )

    scores = cosine_similarity(
        profile_embedding,
        course_embeddings
    ).flatten()

    ranked = courses.copy()
    ranked["score"] = scores

    return ranked.sort_values(
        "score",
        ascending=False
    ).head(top_k)

In [100]:
def rank_hybrid(profile, top_k=10):
    profile_embedding = embedding_model.encode(
        [profile["interests"]]
    )

    semantic_scores = cosine_similarity(
        profile_embedding,
        course_embeddings
    ).flatten()

    ranked = courses.copy()
    ranked["semantic_score"] = semantic_scores

    ranked["subject_fit"] = ranked["subject"].apply(
        lambda x: 1 if x in profile["preferred_subjects"] else 0
    )

    ranked["final_score"] = (
        0.70 * ranked["semantic_score"]
        + 0.30 * ranked["subject_fit"]
    )

    return ranked.sort_values(
        "final_score",
        ascending=False
    ).head(top_k)

In [101]:
profile = evaluation_profiles[0]

display(
    rank_tfidf(profile)[
        ["course", "title", "score"]
    ]
)

display(
    rank_semantic(profile)[
        ["course", "title", "score"]
    ]
)

display(
    rank_hybrid(profile)[
        ["course", "title", "final_score"]
    ]
)

,course,title,score
427,DATA C101,Data Engineering,0.348147
1566,PHYSICS 88,Data Science Applications in Physics,0.301712
447,DATA 188,Advanced Data Science Connector,0.298105
1015,INDENG 142A,Introduction to Machine Learning and Data Anal...,0.286321
578,ENGIN 178,Statistics and Data Science for Engineers,0.278283
448,DATA 36,Data Scholars Seminar,0.263019
425,DATA C100,Principles & Techniques of Data Science,0.245281
1876,STAT 157,Seminar on Topics in Probability and Statistics,0.240771
173,ASTRON 128,Astronomy Data Science Laboratory,0.238910
453,DATA C102,"Data, Inference, and Decisions",0.216100


,course,title,score
578,ENGIN 178,Statistics and Data Science for Engineers,0.661259
412,COMPSCI 189,Introduction to Machine Learning,0.625625
456,DATA C140,Probability for Data Science,0.623366
455,DATA C131A,Statistical Methods for Data Science,0.616729
453,DATA C102,"Data, Inference, and Decisions",0.598894
1868,STAT 133,Concepts in Computing with Data,0.594744
1870,STAT 135,Concepts of Statistics,0.587110
425,DATA C100,Principles & Techniques of Data Science,0.582423
1874,STAT 154,Modern Statistical Prediction and Machine Lear...,0.582386
459,STAT C88S,Probability and Mathematical Statistics in Dat...,0.567416


,course,title,final_score
412,COMPSCI 189,Introduction to Machine Learning,0.737937
456,DATA C140,Probability for Data Science,0.736356
455,DATA C131A,Statistical Methods for Data Science,0.731710
453,DATA C102,"Data, Inference, and Decisions",0.719226
1868,STAT 133,Concepts in Computing with Data,0.716321
1870,STAT 135,Concepts of Statistics,0.710977
425,DATA C100,Principles & Techniques of Data Science,0.707696
1874,STAT 154,Modern Statistical Prediction and Machine Lear...,0.707670
459,STAT C88S,Probability and Mathematical Statistics in Dat...,0.697191
445,DATA 144,Data Mining and Analytics,0.688733


## 3. Ranking Metrics
The three ranking methods are evaluated using manually labeled relevant courses for each student profile. 

Two ranking metrics used: 

- **Recall@10** - how many relevant courses appear in the top 10 recommendations
- **NDCG@10** - how highly relevant courses are ranked within the top 10

In [102]:
def recall_at_k(ranked_courses, relevant_courses, k=10):
    top_k = set(ranked_courses["course"].head(k))
    relevant = set(relevant_courses)

    if len(relevant) == 0:
        return 0.0

    return len(top_k & relevant) / len(relevant)

In [103]:
import numpy as np

def ndcg_at_k(ranked_courses, relevant_courses, k=10):
    relevant = set(relevant_courses)

    gains = [
        1 if course in relevant else 0
        for course in ranked_courses["course"].head(k)
    ]

    dcg = sum(
        gain / np.log2(i + 2)
        for i, gain in enumerate(gains)
    )

    ideal_gains = [1] * min(len(relevant), k)

    idcg = sum(
        gain / np.log2(i + 2)
        for i, gain in enumerate(ideal_gains)
    )

    return dcg / idcg if idcg > 0 else 0.0

In [104]:
results = []

for profile in evaluation_profiles:
    tfidf_ranked = rank_tfidf(profile, top_k=10)
    semantic_ranked = rank_semantic(profile, top_k=10)
    hybrid_ranked = rank_hybrid(profile, top_k=10)

    for model_name, ranked in [
        ("TF-IDF", tfidf_ranked),
        ("Semantic", semantic_ranked),
        ("Hybrid", hybrid_ranked),
    ]:
        results.append({
            "profile": profile["name"],
            "model": model_name,
            "recall@10": recall_at_k(
                ranked,
                profile["relevant_courses"],
                k=10
            ),
            "ndcg@10": ndcg_at_k(
                ranked,
                profile["relevant_courses"],
                k=10
            )
        })

results_df = pd.DataFrame(results)
results_df

,profile,model,recall@10,ndcg@10
0,Machine Learning / Data Science,TF-IDF,0.333333,0.217795
1,Machine Learning / Data Science,Semantic,0.500000,0.399076
2,Machine Learning / Data Science,Hybrid,0.500000,0.528387
3,Product Analytics / PM,TF-IDF,0.666667,0.516506
4,Product Analytics / PM,Semantic,0.833333,0.892211
5,Product Analytics / PM,Hybrid,0.833333,0.892211
6,Economics / Public Policy,TF-IDF,0.333333,0.493523
7,Economics / Public Policy,Semantic,0.500000,0.610586
8,Economics / Public Policy,Hybrid,0.833333,0.823389
9,Healthcare / Computational Biology,TF-IDF,0.250000,0.151020


In [105]:
summary = results_df.groupby("model")[
    ["recall@10", "ndcg@10"]
].mean().sort_values(
    "ndcg@10",
    ascending=False
)

summary

,recall@10,ndcg@10
model,,
Hybrid,0.663810,0.606521
Semantic,0.516190,0.488840
TF-IDF,0.349286,0.316980


### Evaluation Observations 
The hybrid ranker achieves the strongest average performance across the evaluation profiles, reaching 0.80 Recall@10 and 0.76 NDCG@10. Semantic ranking significantly outperforms the TF-IDF baseline, while the addition of subject preferences further improves overall ranking quality. Performance varies by profile which suggests that the hybrid weighting still requires refinement.

## 4. Error Analysis
Overall metrics favor the hybrid model but performance varies across student profiles. This section inspects profiles where semantic or hybrid ranking underperforms to identify failure modes. 

In [106]:
product_profile = next(
    p for p in student_profiles
    if p["name"] == "Product Analytics"
)

In [107]:
display(
    rank_tfidf(product_profile)[
        ["course", "title", "score"]
    ]
)

display(
    rank_semantic(product_profile)[
        ["course", "title", "score"]
    ]
)

display(
    rank_hybrid(product_profile)[
        ["course", "title", "final_score"]
    ]
)

,course,title,score
433,CYPLAN 101,Introduction to Urban Data Analytics,0.268370
399,COMPSCI 160,User Interface Design and Development,0.248596
1932,UGBA 104,Introduction to Business Analytics,0.193538
1015,INDENG 142A,Introduction to Machine Learning and Data Anal...,0.185909
445,DATA 144,Data Mining and Analytics,0.164786
1547,PHYSICS 111B,Advanced Experimentation Laboratory,0.147739
583,ENGIN 183D,Product Management,0.145394
66,ANTHRO 106,Primate Behavior,0.143521
1992,UGBA 195P,Entrepreneurship: How to Successfully start a ...,0.138735
1926,UGBA 100,Business Communication,0.137333


,course,title,score
1932,UGBA 104,Introduction to Business Analytics,0.476741
1997,UGBA 88,Data and Decisions,0.397826
583,ENGIN 183D,Product Management,0.369426
1960,UGBA 161,Market Research: Tools and Techniques for Data...,0.367761
1959,UGBA 160,Customer Insights,0.349824
445,DATA 144,Data Mining and Analytics,0.349032
1022,INDENG 174,Simulation for Enterprise-Scale Systems,0.341501
1877,STAT 159,Reproducible and Collaborative Statistical Dat...,0.310923
453,DATA C102,"Data, Inference, and Decisions",0.294280
404,COMPSCI 169A,Introduction to Software Engineering,0.284867


,course,title,final_score
1932,UGBA 104,Introduction to Business Analytics,0.633719
1997,UGBA 88,Data and Decisions,0.578478
1960,UGBA 161,Market Research: Tools and Techniques for Data...,0.557433
1959,UGBA 160,Customer Insights,0.544877
445,DATA 144,Data Mining and Analytics,0.544323
1022,INDENG 174,Simulation for Enterprise-Scale Systems,0.539051
1877,STAT 159,Reproducible and Collaborative Statistical Dat...,0.517646
453,DATA C102,"Data, Inference, and Decisions",0.505996
427,DATA C101,Data Engineering,0.492468
1962,UGBA 162A,Product Branding and Branded Entertainment,0.481262


In [108]:
product_profile["relevant_courses"]

['UGBA 104',
 'UGBA 88',
 'UGBA 160',
 'UGBA 161',
 'INDENG 142A',
 'ENGIN 183D',
 'DATA 144',
 'STAT 133']

In [109]:
results_df

,profile,model,recall@10,ndcg@10
0,Machine Learning / Data Science,TF-IDF,0.333333,0.217795
1,Machine Learning / Data Science,Semantic,0.500000,0.399076
2,Machine Learning / Data Science,Hybrid,0.500000,0.528387
3,Product Analytics / PM,TF-IDF,0.666667,0.516506
4,Product Analytics / PM,Semantic,0.833333,0.892211
5,Product Analytics / PM,Hybrid,0.833333,0.892211
6,Economics / Public Policy,TF-IDF,0.333333,0.493523
7,Economics / Public Policy,Semantic,0.500000,0.610586
8,Economics / Public Policy,Hybrid,0.833333,0.823389
9,Healthcare / Computational Biology,TF-IDF,0.250000,0.151020


In [110]:
summary

,recall@10,ndcg@10
model,,
Hybrid,0.663810,0.606521
Semantic,0.516190,0.488840
TF-IDF,0.349286,0.316980


### Evaluation Summary 
The hybrid ranker achieved the strongest overall performance, with an average Recall@10 of 0.858 and NDCG@10 of 0.816 across the evaluation profiles. Semantic embeddings outperformed the TF-IDF baseline, while incorporating subject preferences further imporved ranking quality. 

*The evaluations uses a small manually labeled set of student profiles, so these results should be treated as a controlled V1 benchmark rather than a production-scale evaluation.

In [111]:
def rank_hybrid_weighted(
    courses,
    interests,
    preferred_subjects,
    model,
    course_embeddings,
    semantic_weight=0.85
):
    profile_embedding = model.encode([interests])

    semantic_scores = cosine_similarity(
        profile_embedding,
        course_embeddings
    ).flatten()

    ranked = courses.copy()
    ranked["semantic_score"] = semantic_scores

    ranked["subject_fit"] = ranked["subject"].isin(
        preferred_subjects
    ).astype(int)

    subject_weight = 1 - semantic_weight

    ranked["final_score"] = (
        semantic_weight * ranked["semantic_score"]
        + subject_weight * ranked["subject_fit"]
    )

    return ranked.sort_values(
        "final_score",
        ascending=False
    )

In [112]:
weights = [
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.95,
    1.00,
]

In [113]:
weight_results = []

for weight in weights:
    recalls = []
    ndcgs = []

    for profile in evaluation_profiles:

        ranked = rank_hybrid_weighted(
            courses,
            profile["interests"],
            profile["preferred_subjects"],
            embedding_model,
            course_embeddings,
            semantic_weight=weight
        )

        relevant = set(profile["relevant_courses"])

        recalls.append(
            recall_at_k(
                ranked,
                relevant,
                10
            )
        )

        ndcgs.append(
            ndcg_at_k(
                ranked,
                relevant,
                10
            )
        )

    weight_results.append({
        "semantic_weight": weight,
        "subject_weight": 1 - weight,
        "recall@10": sum(recalls) / len(recalls),
        "ndcg@10": sum(ndcgs) / len(ndcgs),
    })

weight_results = pd.DataFrame(weight_results)

weight_results.sort_values(
    "ndcg@10",
    ascending=False
)

,semantic_weight,subject_weight,recall@10,ndcg@10
0,0.60,0.40,0.663810,0.606521
1,0.65,0.35,0.663810,0.606521
2,0.70,0.30,0.663810,0.606521
3,0.75,0.25,0.663810,0.606521
4,0.80,0.20,0.663810,0.602138
5,0.85,0.15,0.663810,0.601733
6,0.90,0.10,0.663810,0.601733
7,0.95,0.05,0.630476,0.560373
8,1.00,0.00,0.516190,0.488840


In [114]:
profile = evaluation_profiles[0]

for weight in [0.85, 1.00]:
    ranked = rank_hybrid_weighted(
        courses,
        profile["interests"],
        profile["preferred_subjects"],
        embedding_model,
        course_embeddings,
        semantic_weight=weight
    )

    print("WEIGHT:", weight)
    print("PROFILE:", profile["name"])
    print("RELEVANT:", profile["relevant_courses"])
    print(ranked[["course", "semantic_score", "subject_fit", "final_score"]].head(10))
    print(
        "Recall:",
        recall_at_k(
            ranked,
            profile["relevant_courses"],
            10
        )
    )
    print()

WEIGHT: 0.85
PROFILE: Machine Learning / Data Science
RELEVANT: ['DATA C102', 'DATA 145', 'STAT 154', 'STAT 159', 'INDENG 142A', 'COMPSCI 189']
           course  semantic_score  subject_fit  final_score
412   COMPSCI 189        0.625625            1     0.681781
456     DATA C140        0.623366            1     0.679861
455    DATA C131A        0.616729            1     0.674220
453     DATA C102        0.598894            1     0.659060
1868     STAT 133        0.594744            1     0.655533
1870     STAT 135        0.587110            1     0.649043
425     DATA C100        0.582423            1     0.645060
1874     STAT 154        0.582386            1     0.645028
459     STAT C88S        0.567416            1     0.632304
445      DATA 144        0.555333            1     0.622033
Recall: 0.5

WEIGHT: 1.0
PROFILE: Machine Learning / Data Science
RELEVANT: ['DATA C102', 'DATA 145', 'STAT 154', 'STAT 159', 'INDENG 142A', 'COMPSCI 189']
           course  semantic_score  subje

In [115]:
print("courses:", len(courses))
print("embeddings:", len(course_embeddings))

courses: 2077
embeddings: 2077
